# Khám phá dữ liệu ảnh

Đặt bộ dữ liệu đã giải nén dưới `data/raw/`. Trên Colab, chép dự án vào ổ làm việc, dùng `%cd` tới thư mục chứa `src/` rồi cài `requirements.txt` như README. Notebook dùng pipeline chung để tạo manifest và báo cáo.

Nếu ảnh giống hệt pixel nằm ở nhiều tập thì ưu tiên test, kế đến validation và train. Khi validation thiếu lớp sau lọc, bổ sung theo lớp từ train sạch với seed 42. Tập test sạch được giữ nguyên trong bước bổ sung. Xem audit CSV và danh sách ảnh được chuyển trong manifest.

In [ ]:
from pathlib import Path
import sys

from IPython.display import Image as NotebookImage, display

project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents) if (folder / "src" / "config.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Chuyển thư mục làm việc về dự án trước. Trên Colab, xem hướng dẫn trong README.")
sys.path.insert(0, str(project_root))

from src.config import Paths
paths = Paths(project_root)

In [ ]:
import pandas as pd

from src.dataset import prepare_dataset

summary = prepare_dataset(paths)
display(pd.Series(summary).to_frame("Giá trị"))
display(pd.read_csv(paths.metrics / "dataset_class_counts.csv"))

In [ ]:
for name in ("class_counts", "splits", "dimensions", "samples"):
    display(NotebookImage(filename=str(paths.figures / f"eda_{name}.png")))

In [ ]:
audit = pd.read_csv(paths.metrics / "dataset_images.csv")
display(audit.loc[audit["status"] != "kept", ["split", "class_name", "path", "status", "detail"]])
display(pd.read_csv(paths.metrics / "dataset_validation_repair.csv"))

So sánh số lượng trước và sau kiểm tra. Quan sát tỷ số lớp train lớn nhất trên lớp nhỏ nhất, ảnh rất lớn và các trường hợp trùng giữa các tập.